### Performing EDA on OLIST Dataset

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as snb

In [81]:
customer_data = pd.read_csv("..\data\olist_customers_dataset.csv")
olist_orders = pd.read_csv("..\data\olist_orders_dataset.csv")
olist_order_items = pd.read_csv("..\data\olist_order_items_dataset.csv")
olist_order_payments = pd.read_csv("..\data\olist_order_payments_dataset.csv")
olist_order_reviews = pd.read_csv("..\data\olist_order_reviews_dataset.csv")
olist_products = pd.read_csv("..\data\olist_products_dataset.csv")
olist_product_category = pd.read_csv("..\data\product_category_name_translation.csv")

In [110]:
customer_data.head()
olist_orders.head()
olist_order_items.head()
# olist_order_payments.head()
# # olist_order_reviews.head()
# olist_products.head()
# olist_product_category.head()


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


### Checking Multiple Payments issue

In [ ]:

counts = olist_order_payments["order_id"].value_counts()
duplicate_ids = counts[counts>1].index
olist_order_payments[olist_order_payments["order_id"].isin(duplicate_ids)].sort_values(by="order_id")

#Faster Way
olist_order_payments[olist_order_payments.duplicated(subset=['order_id'], keep=False)].sort_values(by='order_id')

In [ ]:
# Grouping multiple orders in payments
unique_order_payments = olist_order_payments.groupby(["order_id"]).agg(max_installments=('payment_installments', 'max'),
                                                                       max_methods = ('payment_sequential', 'max'),
                                                                       total_payment = ('payment_value','sum')).reset_index()
primary_payments=olist_order_payments.sort_values(by="payment_value", ascending=False).drop_duplicates(subset="order_id")
primary_payments

# Final Cleaned Payments Data
olist_cleaned_payments = pd.merge(primary_payments[['order_id','payment_type']], unique_order_payments,
                                  on="order_id", how="inner")
olist_cleaned_payments.head(26)

### Merging customer and order data

In [ ]:

customer_order_data = pd.merge(customer_data, olist_orders, how="left", on="customer_id")
customer_order_data.head()

#### pd.merge ignores Index!!

In [108]:
# Merging Customer and Payment data
master_customer_data = pd.merge(customer_order_data, olist_cleaned_payments, on="order_id", how="inner")
master_customer_data.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,payment_type,max_installments,max_methods,total_payment
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-05-16 15:22:12,2017-05-23 10:47:57,2017-05-25 10:35:35,2017-06-05 00:00:00,credit_card,2,1,146.87
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-01-12 20:58:32,2018-01-15 17:14:59,2018-01-29 12:41:19,2018-02-06 00:00:00,credit_card,8,1,335.48
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45,2018-05-20 16:19:10,2018-06-11 14:31:00,2018-06-14 17:58:51,2018-06-13 00:00:00,credit_card,7,1,157.73
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38,2018-03-13 17:29:19,2018-03-27 23:22:42,2018-03-28 16:04:25,2018-04-10 00:00:00,credit_card,1,1,173.30
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30,2018-07-29 10:10:09,2018-07-30 15:16:00,2018-08-09 20:55:48,2018-08-15 00:00:00,credit_card,8,1,252.25


### order_items and product data handling

In [125]:
olist_order_items[olist_order_items.duplicated(subset=["order_id"],keep=False)]
olist_order_items.groupby(["order_id","product_id"]).agg(num_orders=("order_item_id","max")).head(50)

,,num_orders
order_id,product_id,
00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,1
00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,1
000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,1
00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,1
00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,1
00048cc3ae777c65dbb7d2a0634bc1ea,ef92defde845ab8450f9d70c526ef70f,1
00054e8431b9d7675808bcb819fb4a32,8d4f2bb7e93e6710a28f34fa83ee7d28,1
000576fe39319847cbb9d288c5617fa6,557d850972a7d6f792fd18ae1400d9b6,1
0005a1a1728c9d785b8e2b08b904576c,310ae3c140ff94b03219ad0adc3c778f,1
